<a href="https://colab.research.google.com/github/lyudmildimitrov1997-sudo/pppp/blob/main/Case1_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

In [52]:
random_variable = 42

MERGE THE TWO DATASETS AND VERIFY SUPPLIER IDs

In [53]:
data_a = pd.read_excel("cybersecurity_assessment.xlsx")
data_b = pd.read_excel("supplier_master.xlsx")


In [54]:
print("Unique IDs in dataset A:", data_a['supplier_id'].nunique())
print("Unique IDs in dataset B:", data_b['supplier_id'].nunique())
print("Do all IDs match exactly?", set(data_a['supplier_id']) == set(data_b['supplier_id']))

Unique IDs in dataset A: 1200
Unique IDs in dataset B: 1200
Do all IDs match exactly? True


In [55]:
data = pd.merge(left=data_a, right=data_b, on="supplier_id", how="inner")
print("Merged dataset shape:", data.shape)

Merged dataset shape: (1200, 22)


EXPLORE MISSING VALUES AND CLASS DISTRIBUTION

Check percentage of suppliers with a cybersecurity incident

In [56]:
print("\nIncident rate (Class Distribution):")
print(data["cybersecurity_incident_past_12m"].value_counts(normalize=True) * 100)


Incident rate (Class Distribution):
cybersecurity_incident_past_12m
False    79.666667
True     20.333333
Name: proportion, dtype: float64


How many rows have an NaN in any column

In [57]:
print("\nTotal rows with any missing values:", data.isnull().any(axis=1).sum())
print("Missing values details per column:")
print(data.isnull().sum()[data.isnull().sum() > 0])


Total rows with any missing values: 176
Missing values details per column:
security_rating                   30
average_patch_delay_days          41
mfa_coverage_pct                  46
security_training_coverage_pct    31
security_questionnaire_score      42
dtype: int64


Delete rows with empty values

In [58]:
data = data.dropna(axis=0)
print("Shape after dropping rows with missing values:", data.shape)

Shape after dropping rows with missing values: (1024, 22)


DEFINE FEATURES, TARGET AND TRAIN- TEST SPLIT

Define feature columns

numerical baseline predictors

In [59]:
feature_cols = [
    "external_attack_surface_score", "security_rating", "average_patch_delay_days",
    "mfa_coverage_pct", "security_training_coverage_pct", "prior_incidents_3y",
    "fourth_party_dependency_count", "security_questionnaire_score"
]

X = data[feature_cols]
y = data["cybersecurity_incident_past_12m"].astype(int)

Split into Train and Test sets

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_variable)

TEST THE VARIOUS MACHINE LEARNING MODELS

Logistic Regression

In [61]:
log_reg = LogisticRegression(max_iter=1000, random_state=random_variable)
log_reg.fit(X_train, y_train)
pred_lr = log_reg.predict(X_test)
prob_lr = log_reg.predict_proba(X_test)[:, 1]

Decision Tree Classifier

In [62]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(max_depth=5, random_state=random_variable)
dt_clf.fit(X_train, y_train)
pred_dt = dt_clf.predict(X_test)
prob_dt = dt_clf.predict_proba(X_test)[:, 1]

Prediction Model

In [63]:
prediction_model = LogisticRegression(max_iter=1000, random_state=random_variable)
prediction_model.fit(X_train, y_train)
pred_pm = prediction_model.predict(X_test)
prob_pm = prediction_model.predict_proba(X_test)[:, 1]

COMPARE MODELS USING APPROPRIATE METRICS

In [64]:
print("\n" + "="*40)
print("LOGISTIC REGRESSION PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_lr))
print("F1-Score: ", f1_score(y_test, pred_lr))
print("ROC-AUC:  ", roc_auc_score(y_test, prob_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_lr))

print("\n" + "="*40)
print("DECISION TREE PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_dt))
print("F1-Score: ", f1_score(y_test, pred_dt))
print("ROC-AUC:  ", roc_auc_score(y_test, prob_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_dt).T)

print("\n" + "="*40)
print("PREDICTION MODEL PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_pm))
print("F1-Score: ", f1_score(y_test, pred_pm))
print("ROC-AUC:  ", roc_auc_score(y_test, prob_pm))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_pm).T)


LOGISTIC REGRESSION PERFORMANCE:
Accuracy:  0.8
F1-Score:  0.3050847457627119
ROC-AUC:   0.785
Confusion Matrix:
 [[155   5]
 [ 36   9]]

DECISION TREE PERFORMANCE:
Accuracy:  0.7853658536585366
F1-Score:  0.2413793103448276
ROC-AUC:   0.7094444444444444
Confusion Matrix:
 [[154  38]
 [  6   7]]

PREDICTION MODEL PERFORMANCE:
Accuracy:  0.8
F1-Score:  0.3050847457627119
ROC-AUC:   0.785
Confusion Matrix:
 [[155  36]
 [  5   9]]
